# Zadanie 4 — pandas: merge, groupby i dane czasowe

**Czas:** 35–45 minut  
**Poziom:** średni  
**Cel:** połączyć metadane budynków z miesięcznymi odczytami, zbudować raport i wykres.

Obie tabele są zapisane w notebooku. Dane są syntetyczne i obejmują **cztery miesiące**, dlatego wyników nie nazywamy rocznymi.


> **WERSJA DLA PROWADZĄCEGO — kompletne rozwiązanie.**
>
> Uruchom komórki od góry. Kod, wyniki kontrolne i interpretacje są częścią odpowiedzi. Dane są syntetyczne.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

buildings = pd.DataFrame({
    "id": ["A", "B", "C", "D"],
    "type": ["mieszkalny", "mieszkalny", "biurowy", "biurowy"],
    "region": ["północ", "południe", "północ", "południe"],
    "area_m2": [40, 65, 82, 55],
})

months = ["2026-01", "2026-02", "2026-03", "2026-04"]
measurements = {
    "A": [180, 160, 170, 190],
    "B": [293, 260, 275, 300],
    "C": [369, 330, 350, 385],
    "D": [250, 230, 245, 255],
}
readings = pd.DataFrame([
    {"id": building_id, "month": month, "kwh": value}
    for building_id, values in measurements.items()
    for month, value in zip(months, values)
])

print(buildings)
print(readings.head())


## A. Połącz tabele i sprawdź wynik

1. Zmień `readings["month"]` na typ daty i upewnij się, że są cztery unikalne miesiące.
2. Połącz `readings` z `buildings` po `id` do tabeli `data`. Użyj `validate="many_to_one"`.
3. Sprawdź liczbę wierszy, braki po połączeniu i liczbę budynków. Wyjaśnij, dlaczego tutaj relacja ma być »wiele do jednego«.
4. Dodaj `kwh_per_m2 = kwh / area_m2`.


In [ ]:
readings["month"] = pd.to_datetime(readings["month"], format="%Y-%m")
print("Liczba miesięcy:", readings["month"].nunique())

data = readings.merge(buildings, on="id", how="left", validate="many_to_one", indicator=True)
print("Liczba wierszy:", len(data))
print("Braki po łączeniu:")
print(data.isna().sum())
print("Status łączenia:")
print(data["_merge"].value_counts())
assert (data["_merge"] == "both").all()
data = data.drop(columns="_merge")

data["kwh_per_m2"] = data["kwh"] / data["area_m2"]
print("Liczba budynków:", data["id"].nunique())
print(data.head())


## B. Zbuduj trzy raporty

5. `monthly_total`: suma `kwh` w każdym miesiącu, uporządkowana chronologicznie.
6. `type_total`: suma `kwh` według typu budynku.
7. `building_total`: suma `kwh` dla każdego `id`. Połącz ją z metrażem i oblicz intensywność za **całe cztery miesiące**. Który budynek ma największą wartość na m²?

Podpowiedź: `groupby`, `sum`, `reset_index`, `merge`. Nie sumuj miesięcznych wartości `kwh_per_m2` bez wyjaśnienia okresu, którego dotyczy wynik.


In [ ]:
monthly_total = data.groupby("month", sort=True)["kwh"].sum()
type_total = data.groupby("type")["kwh"].sum()

building_total = (
    data.groupby("id", as_index=False)["kwh"].sum()
        .merge(buildings[["id", "area_m2", "type", "region"]],
               on="id", validate="one_to_one")
)
building_total["kwh_per_m2_4m"] = building_total["kwh"] / building_total["area_m2"]
building_total = building_total.sort_values("kwh_per_m2_4m", ascending=False)

print("Suma miesięczna:")
print(monthly_total)
print("\nSuma według typu:")
print(type_total)
print("\nRanking intensywności za cztery miesiące:")
print(building_total[["id", "kwh", "area_m2", "kwh_per_m2_4m"]])
print("Największa intensywność:", building_total.iloc[0]["id"])


## C. Wykres i dwa wnioski

8. Narysuj wykres liniowy miesięcznego `monthly_total` w Matplotlib. Podpisz osie i jednostkę kWh.
9. Napisz dwa zdania: w którym miesiącu suma była najmniejsza oraz dlaczego sam wykres nie dowodzi, że spadek spowodowała temperatura.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(monthly_total.index, monthly_total.values, marker="o", linewidth=2)
ax.set(title="Łączne zużycie energii w czterech miesiącach",
       xlabel="Miesiąc", ylabel="Zużycie [kWh]")
ax.grid(alpha=0.25)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()


### Dwa wnioski prowadzącego

1. Najmniejsza suma wystąpiła w **lutym 2026 r.** i wyniosła **980 kWh**.
2. Sam wykres pokazuje współwystępowanie wartości w czasie, ale nie zawiera danych o temperaturze ani projektu badania przyczynowego. Nie pozwala więc przypisać spadku temperaturze.


## Samokontrola

Uruchom po wykonaniu raportów.


In [ ]:
assert len(data) == 16
assert data["id"].nunique() == 4
assert data["month"].nunique() == 4
assert not data[["type", "region", "area_m2"]].isna().any().any()
assert int(monthly_total.sum()) == 4242
assert int(monthly_total.min()) == 980
assert int(type_total.loc["mieszkalny"]) == 1828
assert int(type_total.loc["biurowy"]) == 2414
print("Kontrola agregacji zakończona pomyślnie.")


## Wyzwanie dodatkowe

Zbuduj tabelę przestawną z miesiącami w wierszach i typami budynków w kolumnach. Narysuj dwie linie na jednym wykresie, a następnie porównaj przebieg dla obu typów.


## Rozwiązanie wyzwania dodatkowego

Tabela przestawna przedstawia miesięczne sumy osobno dla budynków mieszkalnych i biurowych. Każda kolumna daje jedną serię wykresu.


In [ ]:
by_type_month = data.pivot_table(
    index="month", columns="type", values="kwh", aggfunc="sum"
).sort_index()
print(by_type_month)

ax = by_type_month.plot(marker="o", figsize=(9, 4), linewidth=2)
ax.set(title="Zużycie energii według typu budynku",
       xlabel="Miesiąc", ylabel="Zużycie [kWh]")
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()


**Wniosek:** obie serie mają minimum w lutym. Wartości bezwzględne dla grupy biurowej są wyższe, lecz grupa obejmuje budynki o innym łącznym metrażu. Aby porównać efektywność, należy również odnieść zużycie do powierzchni i innych istotnych warunków.
